# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadAli128106/Week-1_A1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row contains all the daily search performance metrics for one specific URL on a single day.
The time window im using is the month of March 2026. I chose this as a mid panel window and it would be light weight. Furthermore, the lastest data of June 2026 will be used for testing.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: day_of_week(feature engineered), gsc_impressions and gsc_avg_position

Label: CTR which is computed in this manner: gsc_clicks/gsc_impressions

Context: report_date, content_hash_id, client_hash_id, gsc_data_available

Excluded: All GA4 colums, because all of them contains data for what happens after the click which is not relevant. Bool flags (client_has_gsc, ga4_data_available, etc.) excluded as features because after filtering to available rows, they're constant — a column with only one value carries no predictive signal.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [14]:
!pip install duckdb -q

In [15]:
%pip -q install duckdb

import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

# metadata-only, near-free — confirms you're pointed at the right partition before spending anything
con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {REL}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").show()

con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {REL}
""").show()

con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available = TRUE)  AS wrong_eq_true,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS correct_is_true,
        COUNT(*) FILTER (WHERE gsc_data_available IS NULL) AS null_rows
    FROM {REL}
""").show()

df_features = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_avg_position,
        gsc_clicks,
        dayofweek(report_date) AS day_of_week,
        gsc_clicks * 1.0 / gsc_impressions AS ctr
    FROM {REL}
    WHERE gsc_data_available IS TRUE
      AND gsc_impressions > 0
""").df()

df_features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────┐
│ report_date │ client_hash_id │ content_hash_id │   n   │
│    date     │    varchar     │     varchar     │ int64 │
├─────────────┴────────────────┴─────────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘

┌─────────┬────────────┬────────────┐
│ n_rows  │  min_date  │  max_date  │
│  int64  │    date    │    date    │
├─────────┼────────────┼────────────┤
│ 9841378 │ 2026-03-01 │ 2026-03-31 │
└─────────┴────────────┴────────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬───────────────┬─────────────────┬───────────┐
│ total_rows │ wrong_eq_true │ correct_is_true │ null_rows │
│   int64    │     int64     │      int64      │   int64   │
├────────────┼───────────────┼─────────────────┼───────────┤
│    9841378 │       3611061 │         3611061 │         0 │
└────────────┴───────────────┴─────────────────┴───────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,gsc_clicks,day_of_week,ctr
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,3.350000,0,0,0.000
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0.000000,0,0,0.000
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,4.928000,1,0,0.008
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,4.000000,0,0,0.000
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,2.272727,0,0,0.000


In [16]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
# Honest version — your real 3 features
X_honest = df_features[['gsc_impressions', 'gsc_avg_position', 'day_of_week']]
y = df_features['ctr']

X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.2, random_state=42)
model_honest = DecisionTreeRegressor(max_depth=8, random_state=42).fit(X_train, y_train)
score_honest = r2_score(y_test, model_honest.predict(X_test))
print("Honest R²:", score_honest)


X_leaky = df_features[['gsc_impressions', 'gsc_avg_position', 'day_of_week', 'gsc_clicks']]
X_train, X_test, y_train, y_test = train_test_split(X_leaky, y, test_size=0.2, random_state=42)
model_leaky = DecisionTreeRegressor(max_depth=8, random_state=42).fit(X_train, y_train)
score_leaky = r2_score(y_test, model_leaky.predict(X_test))
print("Leaky R² (with gsc_clicks):", score_leaky)

print(f"\nJump: {score_honest:.3f} -> {score_leaky:.3f}")


X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.2, random_state=42)
model_honest = DecisionTreeRegressor(max_depth=8, random_state=42).fit(X_train, y_train)
score_honest = r2_score(y_test, model_honest.predict(X_test))
print("Honest R²:", score_honest)


Honest R²: 0.0055840850772719275
Leaky R² (with gsc_clicks): 0.9984016710595076

Jump: 0.006 -> 0.998
Honest R²: 0.0055840850772719275


Added `gsc_clicks` as a feature alongside the 3 honest features
(`gsc_impressions`, `gsc_avg_position`, `day_of_week`).

- Honest R² (3 features): **0.0056**
- With leakage R² (after adding gsc_clicks): **0.998**

I first assumed overfitting, but it was leakage, which is a different problem. My label CTR is calculated as `gsc_clicks / gsc_impressions`. By adding
`gsc_clicks` as a feature (not part of the label calculation itself, a
separate step), I handed the model the exact numerator of its own label.
With `gsc_impressions` already present, the tree reconstructed the ratio
instead of learning anything predictive.

Removed `gsc_clicks`. Honest score (0.0056) is what stays in this notebook.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Unbalanced history: Client data starts at different times, meaning some URLs have years of baseline data while others only have a few weeks.

GSC-only early rows: Because platforms were connected at different times, many early records lack GA4 data entirely.

Window overlaps: Rigid daily tracking severs user search journeys; if a user searches at 11:59 PM and clicks at 12:01 AM, it is recorded as two disconnected events.

No "Why": The data strictly logs visibility and clicks. It cannot tell us if a competitor ran ads or an algorithm update pushed our link down.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.